In [ ]:
# ================================================================
# δ-Normalized Root Influence Field + Newton Flow
# Fixed SageMath/Jupyter Version (Bugfix)
# ================================================================
import matplotlib
# Force the TkAgg backend explicitly
matplotlib.use('TkAgg', force=True)
#%matplotlib Tk
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import Button
from matplotlib.patches import Circle
from sage.all import *
import re
import textwrap

# Ensure a clean slate before running
plt.close('all')

# ================================================================
# SETTINGS
# ================================================================
prec = 1000
RHP = RealField(prec)
CHP = ComplexField(prec)
print_digits = 20
USE_GLOBAL_SCALING = True
MAX_PLOT_RADIUS = RHP('1e8')
GRID_RESOLUTION = 400

# Sage infinity (works with high-precision fields)
INF = infinity   # positive infinity from sage.all

# ================================================================
# INPUT
# ================================================================
def get_polynomial_coeffs():
    pattern = re.compile(r'^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$')
    while True:
        s = input("Enter polynomial coefficients (highest degree first): ").strip()
        if not s:
            print("Error: No input provided.")
            continue
        tokens = s.split()
        coeffs_hp = []
        coeff_strings = []
        for t in tokens:
            t_clean = t.lstrip("0") or "0"
            if t_clean.startswith("."):
                t_clean = "0" + t_clean
            if not pattern.fullmatch(t_clean):
                print(f"Invalid number: '{t}'")
                break
            try:
                coeffs_hp.append(RHP(t_clean))
                coeff_strings.append(t)
            except Exception:
                print(f"Could not convert '{t}'")
                break
        else:
            if all(c == 0 for c in coeffs_hp):
                print("Polynomial cannot be identically zero.")
                continue
            while coeffs_hp and coeffs_hp[0] == 0:
                coeffs_hp.pop(0)
                coeff_strings.pop(0)
            return coeffs_hp, coeff_strings

coeffs_hp, coeff_strings = get_polynomial_coeffs()

# ================================================================
# POLYNOMIAL CONSTRUCTION
# ================================================================
def build_polynomial_from_coeffs(coeffs_hp, RHP, CHP):
    R.<x> = RHP[]
    f = R(0)
    for c in coeffs_hp:
        f = f * x + c
    fC = f.change_ring(CHP)
    return f, fC

f, fC = build_polynomial_from_coeffs(coeffs_hp, RHP, CHP)

# ================================================================
# PRINTING HELPERS
# ================================================================
def polynomial_to_string(coeffs_hp, coeff_strings, var="x"):
    if not coeff_strings:
        return "0"
    terms = []
    n = len(coeff_strings)
    for i, (c_str, c_hp) in enumerate(zip(coeff_strings, coeffs_hp)):
        power = n - i - 1
        if abs(c_hp) < RHP('1e-30'):
            continue
        sign = "-" if c_str.startswith("-") else "+"
        fc_abs = c_str.lstrip("+-")
        if power == 0:
            term = fc_abs
        elif power == 1:
            term = f"{var}" if fc_abs == "1" else f"{fc_abs}*{var}"
        else:
            term = f"{var}^{power}" if fc_abs == "1" else f"{fc_abs}*{var}^{power}"
        terms.append((sign, term))

    if not terms:
        return "0"
    first_sign, first_term = terms[0]
    result = first_term if first_sign == "+" else f"-{first_term}"
    for sign, term in terms[1:]:
        result += f" {sign} {term}"
    return result

poly_str = polynomial_to_string(coeffs_hp, coeff_strings)
print(f"\nPolynomial: f(x) = {poly_str} = 0")

def truncate_polynomial(poly_str, max_len=80, keep_head=3, keep_tail=2):
    if len(poly_str) <= max_len:
        return poly_str
    terms = re.findall(r'[+-]?\s*[^+-]+', poly_str)
    terms = [t.strip() for t in terms if t.strip()]
    if len(terms) <= keep_head + keep_tail:
        return poly_str[:max_len - 3] + "..."
    head = terms[:keep_head]
    tail = terms[-keep_tail:]
    candidate = " ".join(head) + " + ⋯ " + " ".join(tail)
    candidate = candidate.replace("+ -", "- ")
    while len(candidate) > max_len and keep_tail > 1:
        keep_tail -= 1
        tail = terms[-keep_tail:]
        candidate = " + ".join(head) + " + ⋯ + " + " + ".join(tail)
        candidate = candidate.replace("+ -", "- ")
    while len(candidate) > max_len and keep_head > 1:
        keep_head -= 1
        head = terms[:keep_head]
        candidate = " + ".join(head) + " + ⋯ + " + " + ".join(tail)
        candidate = candidate.replace("+ -", "- ")
    if len(candidate) > max_len:
        candidate = candidate[:max_len - 1] + "…"
    return candidate

# ================================================================
# ROOTS
# ================================================================
roots = fC.roots(multiplicities=True) if len(coeffs_hp) > 1 else []

# ================================================================
# HELPERS
# ================================================================
def poly_eval(coeffs, r):
    p = CHP(0)
    for c in coeffs:
        p = p * r + c
    return p

def relative_residual(coeffs, r):
    numerator = abs(poly_eval(coeffs, r))
    denom = sum(abs(c) * abs(r)**(len(coeffs) - i - 1) for i, c in enumerate(coeffs))
    return numerator / denom if denom != 0 else numerator

def compute_local_triplet(f_poly, r, m):
    deriv_poly = f_poly
    for _ in range(m):
        deriv_poly = deriv_poly.derivative()
    alpha = deriv_poly(r) / factorial(m)
    abs_alpha = abs(alpha)
    delta = None if abs_alpha == 0 else abs_alpha ** (-RHP(1) / RHP(m))
    return r, m, delta, alpha

# ================================================================
# PRINT ROOT TABLE
# ================================================================
print(f"\nPolynomial degree: {len(coeffs_hp)-1}")
if roots:
    print("\nLocal Asymptotic Triplets T = (a, m, δ)")
    print(f"{'#':<3} {'Root a':<32} {'m':<4} {'δ':<24} {'Residual':<12}")
    print("-" * 110)
    for i, (r, mult) in enumerate(roots, 1):
        _, _, delta, _ = compute_local_triplet(fC, r, mult)
        delta_str = "∞" if delta is None else str(delta.n(digits=14))
        res = relative_residual(coeffs_hp, r)
        print(f"{i:<3} {str(r.n(digits=print_digits)):<32} {mult:<4} {delta_str:<24} {res:.2e}")

# ================================================================
# IMPROVED GEOMETRIC SCALING
# ================================================================
def compute_field_radius(roots_with_mult, fC, use_global_scaling=True, max_plot_radius=MAX_PLOT_RADIUS):
    if not roots_with_mult:
        return RHP(1)
    centroids_hp = []
    deltas_hp = []
    for r, m in roots_with_mult:
        _, _, delta, _ = compute_local_triplet(fC, r, m)
        centroids_hp.append(abs(r))
        deltas_hp.append(delta if delta is not None else INF)
    if use_global_scaling:
        extents = []
        for c, d in zip(centroids_hp, deltas_hp):
            if d is INF or d == INF:
                continue
            extents.append(c + d)
        R = max(extents + [RHP(1)]) * RHP('1.05') if extents else INF
        if R is INF or R > max_plot_radius:
            print("\n→ Global scaling overflow detected\n→ Falling back to root-focused scaling\n")
            fallback = max(centroids_hp + [RHP(1)]) * RHP('1.5')
            return min(fallback, max_plot_radius)
        return R
    return max(centroids_hp + [RHP(1)]) * RHP('1.5')

# ================================================================
# ROOT FIELD
# ================================================================
def compute_root_field(roots_with_mult, N=GRID_RESOLUTION, use_global_scaling=True):
    centroids = np.array([complex(r) for r, _ in roots_with_mult], dtype=complex)
    mults = np.array([float(m) for _, m in roots_with_mult], dtype=float)
    deltas = np.array([np.inf if delta is None else float(delta) for _, _, delta, _ in [compute_local_triplet(fC, r, m) for r, m in roots_with_mult]], dtype=float)

    R_hp = compute_field_radius(roots_with_mult, fC, use_global_scaling)
    R = float(R_hp) if R_hp is not INF else float(MAX_PLOT_RADIUS)
    print(f"\nPlot radius R = {R:.6e}\n")

    xs = np.linspace(-R, R, N)
    ys = np.linspace(-R, R, N)
    X, Y = np.meshgrid(xs, ys)
    Z = X + 1j * Y

    min_dist = np.full(X.shape, np.inf, dtype=float)
    for a, d in zip(centroids, deltas):
        if np.isfinite(d) and d > 0:
            min_dist = np.minimum(min_dist, np.abs(Z - a) / d)
    dist_field = np.log10(np.clip(min_dist, 1e-30, None))

    log_deriv = np.zeros(Z.shape, dtype=complex)
    EPS = 1e-30
    for a, m_val in zip(centroids, mults):
        dz = Z - a
        safe = np.where(np.abs(dz) < EPS, EPS + 0j, dz)
        log_deriv += m_val / safe

    with np.errstate(divide='ignore', invalid='ignore'):
        V = -1.0 / log_deriv
        mag = np.abs(V)
        mag_safe = np.where(mag > 0, mag, 1.0)
        flow_u = np.real(V) / mag_safe
        flow_v = np.imag(V) / mag_safe

    # Fixed: 'fv' is now correctly 'flow_v'
    return xs, ys, dist_field, flow_u, flow_v, R

# ================================================================
# PLOTTING FUNCTIONS
# ================================================================
def build_root_field_figure(roots_with_mult, fC, poly_str="", use_global_scaling=True, N=GRID_RESOLUTION):
    xs, ys, dist, fu, fv, R = compute_root_field(roots_with_mult, N=N, use_global_scaling=use_global_scaling)
    
    fig, ax = plt.subplots(figsize=(12, 10.5))
    im = ax.imshow(dist, extent=[xs[0], xs[-1], ys[0], ys[-1]], origin='lower', cmap='viridis', aspect='equal')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=r'$\log_{10}(\min_i |z-a_i|/\delta_i)$')
    ax.streamplot(xs, ys, fu, fv, density=1.3, color='black', linewidth=0.55, arrowsize=0.9)

    for r_sage, mult in roots_with_mult:
        r = complex(r_sage)
        _, _, delta, _ = compute_local_triplet(fC, r_sage, mult)
        delta_f = float(delta) if delta is not None else np.inf
        if np.isfinite(delta_f) and 0 < delta_f < R * 2:
            circle = Circle((r.real, r.imag), delta_f, fill=False, color='red', linestyle='--', linewidth=1.5, alpha=0.75)
            ax.add_patch(circle)
        ax.scatter(r.real, r.imag, color='red', s=40 * mult**0.65, edgecolors='black', linewidth=0.8, zorder=5)

    if poly_str:
        full_equation = poly_str.strip()
        if not (full_equation.endswith("= 0") or full_equation.endswith("=0")):
            full_equation += " = 0"
        window_title = truncate_polynomial(full_equation, max_len=75, keep_head=3, keep_tail=1)
        title_equation = truncate_polynomial(full_equation, max_len=180, keep_head=5, keep_tail=2)
        wrapped_eq = textwrap.fill(title_equation, width=80)
        try:
            fig.canvas.manager.set_window_title(window_title)
        except Exception:
            pass
    else:
        wrapped_eq = "Polynomial"

    ax.set_title("δ-Normalized Root Influence Field + Newton Flow\n" + wrapped_eq, fontsize=11, pad=18)
    ax.set_xlabel("Re(z)")
    ax.set_ylabel("Im(z)")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    return fig

# ================================================================
# EXECUTION
# ================================================================
if roots:
    # 1. Build Figure 1
    fig1 = build_root_field_figure(roots, fC, poly_str=poly_str, use_global_scaling=USE_GLOBAL_SCALING, N=GRID_RESOLUTION)

    # 2. Extract NumPy arrays for Figure 2
    roots_np = np.array([complex(r) for r, _ in roots], dtype=complex)
    multiplicities = np.array([int(m) for _, m in roots], dtype=int)

    # 3. Build Figure 2
    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    ax1.axhline(0, color='gray', lw=1)
    ax1.axvline(0, color='gray', lw=1)

    max_extent = 0.0
    max_delta = 0.0
    for r_np, (r_sage, mult) in zip(roots_np, roots):
        ax1.scatter(r_np.real, r_np.imag, color='red', s=12*mult, zorder=5)
        _, _, delta, _ = compute_local_triplet(fC, r_sage, mult)
        delta_float = float(delta) if delta is not None else 0.0
        if delta_float > 0:
            circle = Circle((r_np.real, r_np.imag), radius=delta_float, fill=False, linestyle='--', linewidth=1.5, edgecolor='blue', alpha=0.75)
            ax1.add_patch(circle)
            extent = abs(r_np) + delta_float
            if extent > max_extent: max_extent = extent
            if delta_float > max_delta: max_delta = delta_float

    ax1.set_title("Roots in Complex Plane\n(red dot size ∝ multiplicity | blue dashed = δ)")
    ax1.set_xlabel("Re")
    ax1.set_ylabel("Im")
    ax1.grid(True)

    if len(roots_np) > 0:
        max_root = float(max(np.abs(roots_np)))
        plot_radius = max((max_root + max_delta) * 1.28, 2.5)
        ax1.set_xlim(-plot_radius, plot_radius)
        ax1.set_ylim(-plot_radius, plot_radius)
    ax1.set_aspect('equal')

    # ---- Real Line Plot Pipeline ----
    real_roots = [(r, mult) for (r, mult) in roots if abs(r.imag()) < RHP('1e-12')]
    xmin = float(min(roots_np.real)) - 1.0 if roots_np.size > 0 else -5.0
    xmax = float(max(roots_np.real)) + 1.0 if roots_np.size > 0 else 5.0
    
    x_list = np.linspace(xmin, xmax, 2000).tolist()
    for r, _ in real_roots:
        root_float = float(r.real())
        if not any(abs(root_float - x) < 1e-12 for x in x_list):
            x_list.append(root_float)
    x_list.sort()
    x_vals = np.array(x_list)
    y_vals = np.array([float(f(RHP(xx))) for xx in x_vals])

    if np.max(np.abs(y_vals)) > 1e300:
        y_vals = np.clip(y_vals, -1e300, 1e300)

    ax2.plot(x_vals, y_vals, lw=1, label="f(x)")
    ax2.axhline(0, color='gray', lw=1)
    for r, mult in real_roots:
        ax2.scatter(float(r.real()), 0.0, color='red', s=10*mult, zorder=5)
    ax2.set_title("Polynomial on Real Line (size = multiplicity)")
    ax2.set_xlabel("x")
    ax2.set_ylabel("f(x)")
    ax2.grid(True)
    ax2.legend()
    init_ylim = ax2.get_ylim()

    # ----------------------------- Interactive Widgets -----------------------------
    linthresh = 1.0
    def set_linear(event):
        ax2.set_yscale('linear')
        ax2.set_ylim(init_ylim)
        fig2.canvas.draw_idle()

    def set_symlog(event):
        ax2.set_yscale('symlog', linthresh=linthresh, linscale=1.0)
        fig2.canvas.draw_idle()

    axlinear = plt.axes([0.8, 0.02, 0.1, 0.04])
    axsymlog = plt.axes([0.65, 0.02, 0.1, 0.04])
    b_linear = Button(axlinear, 'Linear')
    b_symlog = Button(axsymlog, 'Symlog')
    b_linear.on_clicked(set_linear)
    b_symlog.on_clicked(set_symlog)

    plt.tight_layout()
    
    # 4. SINGLE FINAL RENDER
    plt.show()

else:
    print("No roots to display.")